# 🚀 Colab LLM Server · Api Code CLI Bridge

**Stack:** `Ollama` → `LiteLLM Proxy` → `Cloudflare Tunnel`  
**Model default:** Hermes-4-14B-BF16-abliterated-i1-GGUF (IQ4_XS, ~7 GB)  
**GPU required:** Nvidia T4 — aktifkan di `Runtime → Change runtime type → T4 GPU`

---

## Cara Kerja

| Komponen | Fungsi |
|---|---|
| **Ollama** | Menjalankan 14B LLM di T4 GPU (16 GB VRAM) |
| **LiteLLM Proxy** | Menerjemahkan format API → Ollama API |
| **Cloudflare Tunnel** | Membuat URL publik HTTPS tanpa butuh akun |

## Langkah Penggunaan

1. ✅ Aktifkan GPU T4: `Runtime → Change runtime type → T4 GPU`
2. ▶️  Jalankan cell di bawah ini dan tunggu (~15 menit pada run pertama)
3. 📋 Salin perintah `export` dari output ke terminal lokal kamu
4. 🖥️ masukan url dan apikey ke env, local akan menggunakan LLM di Colab!

## Pilihan Model

| Model | Keunggulan |
|---|---|
| `Hermes-4-14B-BF16-abliterated-i1-GGUF IQ4_XS` ⭐ | fastest, no refusals |
| `Hermes-4-14B-BF16-abliterated-i1-GGUF Q6_K` | Reasoning lebih baik, general purpose |

---

> ⚠️ **Penting:** Biarkan cell ini tetap running saat menggunakan local LLM
> URL Cloudflare bersifat sementara dan berubah setiap restart.

In [ ]:
import subprocess, time, os, re, requests

# ⚙─ CONFIGURATION ───────────────────────────────
# ▸ Pilih model (uncomment salah satu):

# f947 REKOMENDASI: Coding specialist, abliterated (no refusals)
#MODEL = "hf.co/bartowski/Qwen2.5-Coder-14B-Instruct-abliterated-GGUF:Q6_K"
MODEL = "hf.co/mradermacher/Hermes-4-14B-BF16-abliterated-i1-GGUF:IQ4_XS"
#MODEL = "hf.co/mradermacher/Hermes-4-14B-BF16-abliterated-i1-GGUF:Q6_K"

# f948 Qwen3 generasi terbaru, reasoning lebih baik
# MODEL = "hf.co/richardyoung/Qwen3-14B-abliterated-GGUF:Q6_K"

# f949 DeepSeek-R1 distilled — chain-of-thought terbaik untuk bug kompleks
# MODEL = "hf.co/unsloth/DeepSeek-R1-Distill-Qwen-14B-GGUF:Q6_K"

OLLAMA_PORT = 11434
PROXY_PORT  = 4000
API_KEY     = "sk-colab-local"
# ────────────────────────────────────────────────────────────


# ── Utility Functions ───────────────────────────────────────

def sh(cmd, check=True, quiet=False, capture=False):
    """Run shell command."""
    kw = dict(shell=True, check=check)
    if capture:
        kw.update(capture_output=True, text=True)
    elif quiet:
        kw.update(stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return subprocess.run(cmd, **kw)

def bg(cmd, log=None):
    """Run shell command in background."""
    out = open(log, "w") if log else subprocess.DEVNULL
    return subprocess.Popen(cmd, shell=True, stdout=out, stderr=subprocess.STDOUT)

def wait_http(url, timeout=60, name=""):
    """Poll URL until it responds (< 500 status)."""
    for i in range(timeout):
        try:
            if requests.get(url, timeout=2).status_code < 500:
                print(f"  ✅ {name} is ready")
                return True
        except:
            pass
        if i > 0 and i % 15 == 14:
            print(f"  ⌛ Still waiting for {name}… ({i+1}s)")
        time.sleep(1)
    print(f"  ⚠️  {name} not confirmed after {timeout}s — continuing")
    return False

def section(n, total, title):
    bar = "─" * 64
    print(f"\n{bar}\n  [{n}/{total}]  {title}\n{bar}")


# ── BANNER ────────────────────────────────────────────────
print("╔══════════════════════════════════════════════════════════════╗")
print("║   f680  Colab LLM Server · Api Code CLI Bridge             ║")
print("║   Stack: Ollama → LiteLLM Proxy → Cloudflare Tunnel         ║")
print("╚══════════════════════════════════════════════════════════════╝")

# GPU check
r = sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
       check=False, capture=True)
if r.returncode == 0:
    print(f"\n  f3ae  GPU    : {r.stdout.strip()}")
else:
    print("\n  ⚠️  No GPU — aktifkan T4: Runtime → Change runtime type → T4 GPU")
print(f"  f916  Model  : {MODEL}\n")


# ── 1. INSTALL OLLAMA BINARY ────────────────────────────
section(1, 5, "Install Ollama Binary (via Custom GitHub Link)")

# 1. Pastikan zstd terinstal agar bisa mengekstrak file .tar.zst
sh("apt-get update -qq && apt-get install -y -qq zstd", check=False)

# 2. Bersihkan file sisa sebelumnya agar tidak bentrok
sh("rm -rf ollama.tar.zst /usr/local/bin/ollama /usr/bin/ollama", check=False)

# 3. Unduh dari link spesifik yang kamu berikan
url = "https://github.com/ollama/ollama/releases/download/v0.30.10/ollama-linux-amd64.tar.zst"
print(f"  ⌛ Mengunduh Ollama dari {url} ...")
sh(f"wget -q {url} -O ollama.tar.zst")

# 4. Ekstrak file tar.zst langsung ke direktori /usr
# (Ollama memaketkannya dengan folder bin/ dan lib/ di dalamnya)
sh("tar -C /usr -xaf ollama.tar.zst")

# 5. Cek apakah instalasi berhasil
r = sh("ollama --version", capture=True, check=False)
if r.returncode == 0:
    print(f"  ✅ Ollama installed: {r.stdout.strip()}")
else:
    print("  ⚠️ Gagal diekstrak. (Catatan: Jika error, pastikan tidak ada typo pada v0.30.10 di link tersebut)")


# ── 2. START OLLAMA SERVER ────────────────────────────
section(2, 5, "Start Ollama Server")
ollama_proc = bg(
    "OLLAMA_HOST=0.0.0.0:11434 OLLAMA_ORIGINS='*' ollama serve",
    "/tmp/ollama.log"
)
time.sleep(3)
wait_http(f"http://localhost:{OLLAMA_PORT}", 30, "Ollama")


# ── 3. PULL MODEL ──────────────────────────────
section(3, 5, "Pull Model from HuggingFace (~7 GB)")
print(f"  ⌛ First run takes 10-20 min depending on Colab speed…\n")
sh(f"ollama pull {MODEL}")

# Alias ke nama pendek agar LiteLLM bisa referensikan dengan mudah
with open("/tmp/Modelfile", "w") as f:
    f.write(f"FROM {MODEL}\n")
result = sh("ollama create coder -f /tmp/Modelfile", check=False, capture=True)
if result.returncode == 0:
    print(f"\n  ✅ Model ready, aliased as 'coder'")
else:
    # Fallback: gunakan nama lengkap langsung di LiteLLM
    print(f"\n  ℹ️  Alias tidak dibuat, akan gunakan nama model langsung")


# ── 4. LITELLM PROXY (Anthropic API → Ollama) ───────────────
section(4, 5, "LiteLLM: API → Ollama Bridge")
print("  Installing litellm…")
sh("pip install -q 'litellm[proxy]'")

# Tentukan nama model Ollama yang akan dipakai
# Jika alias 'coder' berhasil dibuat, pakai itu; jika tidak, pakai nama asli HF
ollama_model_name = "character" if result.returncode == 0 else MODEL.replace("hf.co/", "hf.co/")

# Map semua nama model
LLMs_models = [
    "character1",
    MODEL,
]
entries = "\n".join(
    f"  - model_name: {n}\n"
    f"    litellm_params:\n"
    f"      model: ollama/{ollama_model_name}\n"
    f"      api_base: http://localhost:{OLLAMA_PORT}"
    for n in LLMs_models
)
litellm_yaml = f"""
model_list:
{entries}

litellm_settings:
  drop_params: true
  set_verbose: false

general_settings:
  master_key: "{API_KEY}"
"""
with open("/tmp/litellm.yaml", "w") as f:
    f.write(litellm_yaml)

litellm_proc = bg(
    f"litellm --config /tmp/litellm.yaml --port {PROXY_PORT} --host 0.0.0.0",
    "/tmp/litellm.log"
)
time.sleep(6)
wait_http(f"http://localhost:{PROXY_PORT}/health", 45, "LiteLLM proxy")


# ── 5. CLOUDFLARE TUNNEL ────────────────────────────
section(5, 5, "Cloudflare Tunnel → Public HTTPS URL")
sh(
    "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/"
    "cloudflared-linux-amd64 -O /usr/local/bin/cloudflared "
    "&& chmod +x /usr/local/bin/cloudflared"
)
cf_proc = bg(
    f"cloudflared tunnel --url http://localhost:{PROXY_PORT}",
    "/tmp/cloudflared.log"
)

print("  ⌛ Waiting for public URL…")
tunnel_url = None
for _ in range(90):
    time.sleep(1)
    try:
        text = open("/tmp/cloudflared.log").read()
        m = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", text)
        if m:
            tunnel_url = m.group(0)
            break
    except:
        pass


# ── READY ────────────────────────────────────────────
print()
print("╔══════════════════════════════════════════════════════════════╗")
if tunnel_url:
    print("║   ✅  SERVER IS READY                                        ║")
else:
    print("║   ⚠️   SERVER STARTED (no tunnel URL found)                  ║")
print("╚══════════════════════════════════════════════════════════════╝")

if tunnel_url:
    print(f"""
  f517  URL     : {tunnel_url}
  f916  Model   : {MODEL}
  f511  API Key : {API_KEY}

  ╔══════════════════════════════════════════════════════════════╗
  ║   Paste di ENV LOKAL kamu untuk pakai Custom LLMs      :     ║
  ╠══════════════════════════════════════════════════════════════╣
  ║                                                              ║
  ║   BASE_URL="{tunnel_url}"
  ║   API_KEY="{API_KEY}"
  ║                                                              ║
  ║                                                              ║
  ╠══════════════════════════════════════════════════════════════╣
  ║   One-liner version:                                         ║
  ╠══════════════════════════════════════════════════════════════╣
  ║                                                              ║
  ║   BASE_URL="{tunnel_url}"
  ║   API_KEY="{API_KEY}"                                        ║
  ║                                                              ║
  ╚══════════════════════════════════════════════════════════════╝

  ⚠️  PENTING:
  • Biarkan cell ini tetap RUNNING saat menggunakan custom LLMs
  • URL bersifat SEMENTARA — berubah setiap kali restart
  • Colab gratis timeout ~90 menit idle
  • Request pertama butuh 30-60 detik (model warm-up)
""")
else:
    print(f"\n  ⚠️  Tidak dapat mendapatkan URL tunnel.")
    print(f"     Cek log: !cat /tmp/cloudflared.log")
    print(f"     LiteLLM proxy lokal: http://localhost:{PROXY_PORT}")

print("  ⌛ Cell berjalan terus menerus. Stop cell untuk shutdown.\n")


# ── KEEP ALIVE (Heartbeat) ─────────────────────────────
tick = 0
try:
    while True:
        time.sleep(60)
        tick += 1
        ts = time.strftime("%H:%M:%S")
        try:
            requests.get(f"http://localhost:{PROXY_PORT}/health", timeout=5)
            status = "f49a healthy"
        except:
            status = "⚠️  unreachable"
        print(f"  [{ts}] heartbeat #{tick:04d} | proxy {status} | {tunnel_url or 'no tunnel'}")
except KeyboardInterrupt:
    print("\n  ⛔ Shutting down all services…")
    for proc in [cf_proc, litellm_proc, ollama_proc]:
        try:
            proc.terminate()
        except:
            pass
    print("  ✅ All services stopped.")